In [1]:
import pandas as pd


In [5]:
df = pd.read_csv(r"C:\Users\bunyo\OneDrive\Desktop\AI_Course\ModularProgramProjects\finalProject\data\sampled_data\sampled_data.csv")

In [6]:
df.head()

,Efficiency,Weight_kg,Acceleration(0-100),Firth_stop_range_km,Battery_kWh,Fast_charger(kW),Towing_kg,Cargo_volume,Price/Range(km),FastCharge_time_hrs,Energy_weight_ratio,Added_Range_1Stop,Range_km_level,Price_level
0,0.473118,0.577071,0.343137,0.418936,0.539171,0.272727,0.400000,0.2750,0.050938,0.443576,0.583782,0.366935,1,1
1,0.408602,0.612235,0.171569,0.656291,0.649770,0.630303,0.666667,0.3335,0.079088,0.223483,0.697531,0.741935,2,1
2,0.344086,0.677264,0.200980,0.690013,0.751152,0.427273,0.250000,0.2150,0.067024,0.372340,0.764383,0.463710,2,2
3,0.543011,0.479769,0.372549,0.351492,0.612903,0.151515,0.000000,0.2335,0.049598,0.885938,0.781900,0.096774,1,1
4,0.392473,0.557322,0.058824,0.619974,0.603687,0.593939,0.000000,0.1855,0.120643,0.223214,0.688162,0.709677,1,2


In [21]:
df = df.dropna()
x = df.drop(columns=['Range_km_level','Price_level'])
y = df[['Range_km_level','Price_level']]

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.multioutput import MultiOutputClassifier
from xgboost import XGBClassifier

# 1. Tayanch model va MultiOutput wrapper'ni yaratamiz
base_model = XGBClassifier(random_state=42)
model = MultiOutputClassifier(base_model)

# 2. Parametrlar tarmog'i (Grid) - DIQQAT: estimator__ prefiksiga e'tibor bering!
param_grid = {
    'estimator__n_estimators': [50, 100, 200],    # Daraxtlar soni
    'estimator__max_depth': [3, 5, 7],            # Daraxtning maksimal chuqurligi
    'estimator__learning_rate': [0.01, 0.1, 0.2]  # O'rganish tezligi
}

# 3. GridSearchCV ni sozlaymiz
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,                 # 3-fold cross validation (ma'lumotni 3 ga bo'lib tekshiradi)
    n_jobs=-1,            # Barcha protsessor yadrolarini ishga tushirish (tezroq ishlashi uchun)
    verbose=2             # Jarayon qanday ketayotganini konsolda ko'rsatib turadi
)

# 4. Tuning jarayonini boshlaymiz (bu biroz vaqt olishi mumkin)
grid_search.fit(x, y)

# 5. Eng yaxshi natijalarni chop etamiz
print("Eng yaxshi parametrlar:", grid_search.best_params_)
print("Eng yuqori aniqlik (Accuracy):", grid_search.best_score_)

# 6. Eng zo'r modelni alohida o'zgaruvchiga olamiz (keyinchalik SHAP uchun kerak bo'ladi)
best_model = grid_search.best_estimator_

Fitting 3 folds for each of 27 candidates, totalling 81 fits
Eng yaxshi parametrlar: {'estimator__learning_rate': 0.2, 'estimator__max_depth': 3, 'estimator__n_estimators': 200}
Eng yuqori aniqlik (Accuracy): 0.900839054157132


In [22]:
import optuna
import numpy as np
from sklearn.multioutput import MultiOutputClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

# 1. Maqsad (Objective) funksiyasini yaratamiz
def objective(trial):
    # A) Optuna qaysi parametrlarni qaysi oraliqda qidirishini belgilaymiz
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        # log=True qilsak, 0.01, 0.1 kabi kichik sonlarni yaxshiroq qidiradi
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True), 
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'eval_metric': 'mlogloss' # Multiclass uchun kerakli metrika
    }
    
    # B) Tayanch modelni chaqiramiz va parametrlarni beramiz
    base_model = XGBClassifier(**param)
    
    # C) MultiOutput wrapper'ga solamiz
    model = MultiOutputClassifier(base_model)
    
    # D) Modelni baholaymiz (Cross Validation orqali)
    # cv=3 ma'lumotni 3 ga bo'lib o'qitadi. n_jobs=-1 barcha yadrolarni ishlatadi
    scores = cross_val_score(model, x, y, cv=3, scoring='accuracy', n_jobs=-1)
    
    # E) O'rtacha aniqlikni (accuracy) qaytaramiz
    return np.mean(scores)

# 2. Optuna "Study" (Tadqiqot) yaratamiz
# Maqsadimiz aniqlikni (accuracy) oshirish bo'lgani uchun direction='maximize'
study = optuna.create_study(direction='maximize', study_name="EV_MultiOutput_Tuning")

# 3. Tuning jarayonini boshlaymiz
# n_trials=20 degani, Optuna 20 xil kombinatsiyani sinab ko'radi
print("Optuna qidiruvni boshladi...")
study.optimize(objective, n_trials=20)

# 4. Natijalarni ko'rish
print("\nEng yaxshi parametrlar:", study.best_params)
print("Eng yuqori aniqlik (Accuracy):", study.best_value)

Optuna qidiruvni boshladi...


[W 2026-03-14 23:56:59,652] Trial 0 failed with parameters: {'n_estimators': 125, 'max_depth': 3, 'learning_rate': 0.13199180622253037, 'subsample': 0.8298182407804013, 'colsample_bytree': 0.8608612957082424} because of the following error: The value nan is not acceptable.
[W 2026-03-14 23:56:59,656] Trial 0 failed with value np.float64(nan).
[W 2026-03-14 23:57:03,300] Trial 1 failed with parameters: {'n_estimators': 226, 'max_depth': 4, 'learning_rate': 0.023706386254228357, 'subsample': 0.7359729858883425, 'colsample_bytree': 0.7138425161788255} because of the following error: The value nan is not acceptable.
[W 2026-03-14 23:57:03,302] Trial 1 failed with value np.float64(nan).
[W 2026-03-14 23:57:06,559] Trial 2 failed with parameters: {'n_estimators': 243, 'max_depth': 7, 'learning_rate': 0.1281214662527909, 'subsample': 0.7231544340563268, 'colsample_bytree': 0.7954687636962583} because of the following error: The value nan is not acceptable.
[W 2026-03-14 23:57:06,561] Trial 2 

ValueError: No trials are completed yet.

In [ ]:
import pandas as pd
import numpy as np
import shap
import optuna
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, make_scorer

# 1. Ma'lumotlarni tayyorlash
df = pd.read_csv('sampled_data.csv').dropna() # Bo'sh qiymatlar yo'qligiga ishonch hosil qilamiz

X = df.drop(columns=['Range_km_level', 'Price_level'])
y = df[['Range_km_level', 'Price_level']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Maxsus baholash funksiyasi (Custom Scorer)
def multioutput_accuracy(y_true, y_pred):
    if hasattr(y_true, 'values'):
        y_true = y_true.values
    if hasattr(y_pred, 'values'):
        y_pred = y_pred.values
        
    accuracies = []
    # Ikkala ustun (Range va Price) uchun aniqlikni hisoblaymiz
    for i in range(y_true.shape[1]):
        acc = accuracy_score(y_true[:, i], y_pred[:, i])
        accuracies.append(acc)
    return np.mean(accuracies)

custom_scorer = make_scorer(multioutput_accuracy)

# 3. Optuna maqsadi
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'random_state': 42
    }
    
    base_model = XGBClassifier(**params)
    model = MultiOutputClassifier(base_model)
    
    # DIQQAT: n_jobs=1 (parallel ishlashni o'chiramiz xatolikni oldini olish uchun)
    # error_score='raise' (agar xato bo'lsa uni yashirmaydi, ochiq ko'rsatadi)
    scores = cross_val_score(
        model, 
        X_train, 
        y_train, 
        cv=3, 
        scoring=custom_scorer, 
        n_jobs=1,              # <--- O'ZGARISH
        error_score='raise'    # <--- O'ZGARISH
    )
    return np.mean(scores)

# Optunani ishga tushirish
print("Optuna qidiruvni boshladi...")
optuna.logging.set_verbosity(optuna.logging.WARNING) 
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

print("\nEng yaxshi parametrlar:", study.best_params)
print("Eng yuqori aniqlik:", study.best_value)

# 4. Yakuniy Model va SHAP
best_params = study.best_params
best_params['random_state'] = 42

final_base_model = XGBClassifier(**best_params)
final_model = MultiOutputClassifier(final_base_model)

final_model.fit(X_train, y_train)
print("\nYakuniy model o'qitildi! SHAP chizilmoqda...")

explainer = shap.TreeExplainer(final_model.estimators_[0])
shap_values = explainer(X_test)

# Range_km_level ning 0-klassi uchun tahlil (siz buni 1 yoki 2 ga o'zgartirishingiz mumkin)
target_class = 0 
shap.plots.waterfall(shap_values[0, :, target_class])

--- GridSearchCV boshlandi ---

--- Optuna boshlandi ---


[W 2026-03-14 23:52:15,261] Trial 0 failed with parameters: {'n_estimators': 180, 'max_depth': 6, 'learning_rate': 0.022089258550233637, 'subsample': 0.9855871917633511} because of the following error: ValueError('multiclass-multioutput is not supported').
joblib.externals.loky.process_executor._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "c:\Users\bunyo\OneDrive\Desktop\AI_Course\ModularProgramProjects\finalProject\myEnv\Lib\site-packages\joblib\externals\loky\process_executor.py", line 490, in _process_worker
    r = call_item()
  File "c:\Users\bunyo\OneDrive\Desktop\AI_Course\ModularProgramProjects\finalProject\myEnv\Lib\site-packages\joblib\externals\loky\process_executor.py", line 291, in __call__
    return self.fn(*self.args, **self.kwargs)
           ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\bunyo\OneDrive\Desktop\AI_Course\ModularProgramProjects\finalProject\myEnv\Lib\site-packages\joblib\parallel.py", line 607, in __call__
    return [func(*args

ValueError: multiclass-multioutput is not supported

In [19]:
y.info()

<class 'pandas.DataFrame'>
RangeIndex: 1311 entries, 0 to 1310
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Range_km_level  1311 non-null   int64
 1   Price_level     1311 non-null   int64
dtypes: int64(2)
memory usage: 20.6 KB
